In [6]:
import pandas as pd
import unicodedata as unicode
import re

def remover_acentos(texto):
    if pd.isna(texto):
        return texto
    return unicode.normalize('NFKD', str(texto)).encode('ASCII', 'ignore').decode('ASCII')

def converter_data_br_para_iso(val):
    if pd.isna(val) or str(val).strip().lower() in ['', 'nan', 'none', 'null', '-']:
        return ''
    
    val_str = str(val).strip().split(' ')[0]
    
    # Se já estiver em YYYY-MM-DD
    if re.match(r'^\d{4}-\d{2}-\d{2}$', val_str):
        return val_str
        
    # Converte DD/MM/YYYY para YYYY-MM-DD
    match = re.match(r'^(\d{1,2})[/-](\d{1,2})[/-](\d{4})$', val_str)
    if match:
        dia, mes, ano = match.groups()
        return f"{ano}-{int(mes):02d}-{int(dia):02d}"
        
    return ''

def limpar_moeda(valor):
    if pd.isna(valor):
        return '0.00'
    
    valor_str = str(valor).replace('R$', '').strip()
    
    # Traços ou vazios em moedas viram 0.00
    if valor_str in ['-', '--', '', 'N/A', 'nan', 'null', 'None']:
        return '0.00'
        
    valor_str = valor_str.replace('.', '')   # Remove ponto de milhar
    valor_str = valor_str.replace(',', '.')  # Troca vírgula por ponto
    
    return valor_str

# 1. Leitura
df = pd.read_csv('../base_csv/original/movimentacao.csv', encoding='latin1', sep=';')

# 2. Cabeçalho limpo (PREVISÃO_ENTREGA -> previsao_entrega)
df.columns = [remover_acentos(col).lower().strip() for col in df.columns]

# 3. Tratamento de DATAS (DD/MM/YYYY -> YYYY-MM-DD)
colunas_data = ['data_pedido', 'data_faturamento', 'previsao_entrega', 'recebido_em']
for col in colunas_data:
    if col in df.columns:
        df[col] = df[col].astype(str).apply(converter_data_br_para_iso)

# 4. Tratamento de MOEDAS
colunas_moeda = ['custo', 'valor_frete', 'custo_total']
for col in colunas_moeda:
    if col in df.columns:
        df[col] = df[col].apply(limpar_moeda)

# NOTA: 'quantidade' e 'total_recebido' NÃO PASSAM POR TRATAMENTO.
# Ficam exatamente com os valores inteiros da planilha original.

# 5. Exportação
df.to_csv('../base_csv/tratado/movimentacao_tratado.csv', index=False, sep=';', encoding='utf-8-sig', na_rep='')

print("Arquivo processado com sucesso!")

Arquivo processado com sucesso!
